In [1]:
import xarray as xr
import pandas as pd
import numpy as np

In [24]:
ALLOWED_CTH_DEV=1000
def _check_val_seviri_coincidence(row):
    points_of_interest = row["val_cth_hist"] > 0
    if type(row["cth_hist"]) == float:
            N+=1
            return np.nan 
    cth_deviations = np.abs(row["cth_hist"][points_of_interest] - row["val_cth_hist"][points_of_interest])
    if cth_deviations.size == 0:
        return False
    mean_height_dev = cth_deviations.mean()  
    if mean_height_dev > ALLOWED_CTH_DEV:
        return False
    return True

def _get_val_seviri_coincidence(row):
    if "val_seviri_coincidence" not in row.keys():
        return _check_val_seviri_coincidence(row)
    else:
        return row["val_seviri_coincidence"]

def _identify_times_to_process(row):
    val_if_hist = row.val_ice_frac_hist
    idx = np.where((val_if_hist >= -0.01) & (val_if_hist <= 1.01))[0]
    return row.track_start_time+idx*pd.Timedelta(minutes=15)



In [25]:
cloud_data = pd.read_parquet("/nfs/ch4/wolke_scratch/dnikolo/GTE_Validation/DARDAR/Test_bulk/Results/Agg_03_glaciations_20070101.0000_20070115.0000.parquet")
cloud_data["val_seviri_coincidence"] = cloud_data.apply(_get_val_seviri_coincidence,axis=1)


In [26]:
valid_clouds = cloud_data[cloud_data["val_seviri_coincidence"]==True]


In [27]:
valid_clouds["colocation_time"]=valid_clouds.apply(_identify_times_to_process, axis=1)

/cluster/work/climate/dnikolo/dump/ipykernel_1789182/1392821060.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_clouds["colocation_time"]=valid_clouds.apply(_identify_times_to_process, axis=1)


In [28]:
row_temp = valid_clouds.iloc[1]
# for col_time in row["colocation_time"]:
col_time = row_temp["colocation_time"][0]
dt = pd.Timestamp(col_time)
time_str = dt.strftime("%Y%m%d_%H%M")

path = f"/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_{time_str}.nc"
print(path)
ds = xr.open_dataset(path)

lat_bins = ds.lat_bin.values
lon_bins = ds.lon_bin.values

lat0 = lat_bins[0]
lon0 = lon_bins[0]

dlat = lat_bins[1] - lat_bins[0]
dlon = lon_bins[1] - lon_bins[0]

nlat = len(lat_bins)
nlon = len(lon_bins)

/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070106_1000.nc


In [29]:
def get_colocation_ind(row):
    lat_idx_list = []
    lon_idx_list = []
    for lat, lon in zip(row.val_intersec_lat,row.val_intersec_lon):
        lat_idx = np.rint((lat - lat0) / dlat).astype(np.int64)
        lon_idx = np.rint((lon - lon0) / dlon).astype(np.int64)

        valid = (
            (lat_idx >= 0) & (lat_idx < nlat) &
            (lon_idx >= 0) & (lon_idx < nlon)
        )
        lat_idx_list.append(lat_idx[valid])
        lon_idx_list.append(lon_idx[valid])
    return lat_idx_list,lon_idx_list
# col_time = row["colocation_time"][0]
def get_bulk_values(row):
    iwc_means  = []
    for i in range(len(row["colocation_time"])):
        col_time = row["colocation_time"][i]
        dt = pd.Timestamp(col_time)
        time_str = dt.strftime("%Y%m%d_%H%M")

        path = f"/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_{time_str}.nc"
        print(path)
        ds = xr.open_dataset(path)
        cb_iwc_values = ds["cb_iwc_mean"].isel(
            lat_bin=xr.DataArray(row["lat_bin_idx"][i], dims="pixel"),
            lon_bin=xr.DataArray(row["lon_bin_idx"][i], dims="pixel"),
            )
        iwc_means.append(cb_iwc_values.mean(axis=2)[0])
    return iwc_means


def extract_iwc_relative_to_peak(row):
    records = []

    track_start = pd.Timestamp(row["track_start_time"])
    peak_ind = int(row["Glac_peak_ind"])

    for i, iwc_profile in enumerate(row["iwc_bulk_means"]):
        col_time = pd.Timestamp(row["colocation_time"][i])

        # Index of the colocation time within the cloud track
        col_ind = int(round((col_time - track_start).total_seconds() / (15 * 60)))

        # Time index relative to the ice-fraction peak
        rel_ind = col_ind - peak_ind

        records.append({
            "Cloud_ID": row["Cloud_ID"],
            "rel_ind": rel_ind,
            "rel_time_min": rel_ind * 15,
            "iwc_profile": np.asarray(iwc_profile)
        })

    return records



In [30]:
    
valid_clouds[["lat_bin_idx", "lon_bin_idx"]] = (
    valid_clouds
    .apply(get_colocation_ind, axis=1, result_type="expand")
)

/cluster/work/climate/dnikolo/dump/ipykernel_1789182/1847284557.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_clouds[["lat_bin_idx", "lon_bin_idx"]] = (
/cluster/work/climate/dnikolo/dump/ipykernel_1789182/1847284557.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_clouds[["lat_bin_idx", "lon_bin_idx"]] = (


In [31]:
valid_clouds["iwc_bulk_means"] = valid_clouds.apply(get_bulk_values, axis=1)

/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070106_1000.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070113_0315.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070103_0930.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070103_1415.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070104_0000.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070104_1645.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070106_1145.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070107_1215.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070107_2200.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070107_1400.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070107_2330.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070108_1300.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070108_2245.nc
/nfs/ch4/wolke_scratch/dn

/cluster/work/climate/dnikolo/dump/ipykernel_1789182/3901068200.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_clouds["iwc_bulk_means"] = valid_clouds.apply(get_bulk_values, axis=1)


In [32]:

all_records = []

for _, row in valid_clouds.iterrows():
    all_records.extend(extract_iwc_relative_to_peak(row))

iwc_rel_df = pd.DataFrame(all_records)
iwc_stats_by_rel_time = (
    iwc_rel_df
    .groupby("rel_ind")["iwc_profile"]
    .agg(
        n_hits="count",
        mean_iwc_profile=lambda profiles: np.nanmean(np.stack(profiles), axis=0),
    )
    .reset_index()
)
iwc_stats_by_rel_time["rel_time_min"] = iwc_stats_by_rel_time["rel_ind"] * 15

/cluster/work/climate/dnikolo/dump/ipykernel_1789182/1348765763.py:12: RuntimeWarning: Mean of empty slice
  mean_iwc_profile=lambda profiles: np.nanmean(np.stack(profiles), axis=0),


In [33]:
iwc_rel_df.to_parquet("iwc_rel_df.parquet")

In [34]:
iwc_stats_by_rel_time

,rel_ind,n_hits,mean_iwc_profile,rel_time_min
0,-130,1,"[6.6554094e-05, 8.0481404e-05, 9.401955e-05, 0...",-1950
1,-118,1,"[4.0261616e-06, 4.802284e-06, 5.61996e-06, 6.4...",-1770
2,-109,1,"[1.4777113e-05, 1.4730823e-05, 1.4769016e-05, ...",-1635
3,-107,1,"[1.958022e-05, 1.881847e-05, 1.8154837e-05, 1....",-1605
4,-86,1,"[1.2987001e-06, 1.2782384e-06, 1.2646836e-06, ...",-1290
...,...,...,...,...
87,84,2,"[2.269367e-05, 2.4204248e-05, 2.6519978e-05, 3...",1260
88,85,2,"[2.1333349e-06, 2.148638e-06, 2.2877994e-06, 2...",1275
89,96,1,"[3.9521245e-05, 4.2748434e-05, 4.735445e-05, 5...",1440
90,137,1,"[4.7486647e-06, 5.263138e-06, 5.4757033e-06, 5...",2055


In [35]:
iwc_stats_by_rel_time.to_parquet("mean_iwc_by_height_data.parquet")

In [38]:
iwc_stats_by_rel_time[abs(iwc_stats_by_rel_time["rel_ind"])<=3]

,rel_ind,n_hits,mean_iwc_profile,rel_time_min
43,-3,2,"[6.513413e-05, 7.851791e-05, 8.1948914e-05, 2....",-45
44,-2,4,"[3.8127066e-05, 4.1219966e-05, 4.490288e-05, 4...",-30
45,-1,3,"[2.3578557e-05, 2.7314845e-05, 3.0500567e-05, ...",-15
46,0,4,"[2.005987e-05, 2.2878101e-05, 2.6293299e-05, 2...",0
47,1,1,"[4.042303e-06, 4.468091e-06, 5.1151233e-06, 5....",15
48,2,1,"[1.3528091e-05, 1.3869065e-05, 1.4130149e-05, ...",30
49,3,1,"[1.911033e-06, 1.9308306e-06, 1.5375213e-06, 1...",45


: 

In [ ]:
valid_clouds.keys()

: 

In [ ]:
valid_clouds["iwc_bulk_means"] = valid_clouds.apply(get_bulk_values, axis=1)

: 

In [ ]:
valid_clouds.iloc[0]["iwc_bulk_means"]

: 

: 

In [ ]:
len(valid_clouds)

: 

In [ ]:
np.empty(len(valid_clouds),5)

: 